<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/04_evaluation_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 784, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 784 (delta 11), reused 1 (delta 1), pack-reused 765 (from 1)
Receiving objects: 100% (784/784), 833.18 KiB | 23.14 MiB/s, done.
Resolving deltas: 100% (513/513), done.


In [2]:
%cd /content/ML-Tech

/content/ML-Tech


In [3]:
!git pull origin main

From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.3 MB/s eta 0:00:00


In [5]:
import faiss
index = faiss.read_index("data/processed/passport_index.faiss")

In [6]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [7]:
import numpy as np
import pandas as pd
import time

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks:", len(chunks))

Chunks: 35


In [11]:
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [13]:
index = faiss.read_index(
    "data/processed/passport_index.faiss"
)

print("FAISS vectors:", index.ntotal)
print("Chunks:", len(chunks))

assert index.ntotal == len(chunks)

FAISS vectors: 35
Chunks: 35


In [14]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "service":chunks[idx].get("service",""),
            "section": chunks[idx]["section"],
            "language":chunks[idx].get("language",""),
            "text": chunks[idx]["text"],
            "url": chunks[idx]["url"]
        })

    return results

In [15]:
results = retrieve(
    "How much does a 10-year passport cost?",
    k=3
)

for r in results:
    print(
        r["score"],
        "|",
        r["service"],
        "|",
        r["section"]
    )

0.848783016204834 | Biometric Passport | Fees
0.837473452091217 | Biometric Passport | Requested documents
0.8267394304275513 | Biometric Passport | NB


In [18]:
test_questions = [
    {
        "question": "How much does a 10-year passport cost?",
        "expected_service": "Biometric Passport",
        "expected_section": "Fees"
    },

    {
        "question": "What should I do if I lose my passport?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Lost Passport"
    },

    {
        "question": "ما هي المستندات المطلوبة لتجديد رخصة سوق منتهية الصلاحية؟",
        "expected_service": "تجديد رخصة سوق منتهية الصلاحة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "question": "How much is it?",
        "expected_service": "Biometric Passport",
        "expected_section": "Fees"
    },
    {
        "question": "I lost my travel document, what now?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Lost Passport"
    },
    {
        "question": "What papers do I need if my passport was taken?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Stolen Passport"
    },
    {
        "question": "شو بعمل إذا ضاع مني دفتر السوق؟",
        "expected_service": "تجديد رخصة قيادة ضائعة",
        "expected_section": "الوصف"
    },
    {
        "question": "عندي رخصة أجنبية وبدي حولها للبنانية",
        "expected_service": "استبدال رخصة سوق أجنبية برخصة سوق لبنانية",
        "expected_section": "الوصف"
    }
]

In [19]:
def evaluate_retrieval(test_questions, k=3):
    top1_correct = 0
    recall_k_correct = 0
    reciprocal_ranks = []

    details = []

    for test in test_questions:
        results = retrieve(test["question"], k=k)

        expected_service = test["expected_service"].strip()
        expected_section = test["expected_section"].strip()

        rank_found = None

        for rank, result in enumerate(results, start=1):
            service_match = (
                result.get("service", "").strip() == expected_service
            )

            section_match = (
                result.get("section", "").strip() == expected_section
            )

            if service_match and section_match:
                rank_found = rank
                break

        if rank_found == 1:
            top1_correct += 1

        if rank_found is not None:
            recall_k_correct += 1
            reciprocal_ranks.append(1 / rank_found)
        else:
            reciprocal_ranks.append(0)

        details.append({
            "question": test["question"],
            "rank_found": rank_found,
            "top_result_service": results[0].get("service", ""),
            "top_result_section": results[0].get("section", "")
        })

    n = len(test_questions)

    return {
        "top1_accuracy": top1_correct / n,
        f"recall@{k}": recall_k_correct / n,
        "mrr": sum(reciprocal_ranks) / n,
        "details": details
    }

In [20]:
metrics = evaluate_retrieval(test_questions, k=3)

print("Top-1 Accuracy:", metrics["top1_accuracy"])
print("Recall@3:", metrics["recall@3"])
print("MRR:", metrics["mrr"])

Top-1 Accuracy: 0.75
Recall@3: 0.75
MRR: 0.75


In [21]:
answerable_questions = [

    # =========================================================
    # ENGLISH - PASSPORT
    # =========================================================

    {
        "id": 1,
        "language": "en",
        "question": "What documents do I need to apply for a biometric passport?",
        "expected_answer": (
            "The applicant needs the passport application, a Lebanese ID card "
            "or civil status extract, a new certified color photo, the old passport "
            "if available with copies of its non-empty pages, and the required fees."
        )
    },

    {
        "id": 2,
        "language": "en",
        "question": "How much does a 10-year Lebanese biometric passport cost?",
        "expected_answer": "10,000,000 L.L."
    },

    {
        "id": 3,
        "language": "en",
        "question": "How much does a 5-year Lebanese biometric passport cost?",
        "expected_answer": "6,000,000 L.L."
    },

    {
        "id": 4,
        "language": "en",
        "question": "What should I do if I lose my passport?",
        "expected_answer": (
            "The loss must be declared in person at a regional General Security office "
            "with identification documents."
        )
    },

    {
        "id": 5,
        "language": "en",
        "question": "What additional document is required if my passport was stolen?",
        "expected_answer": (
            "A certified copy of the investigation report prepared by the "
            "Internal Security Forces (ISF)."
        )
    },

    {
        "id": 6,
        "language": "en",
        "question": "Do I have to report a stolen passport if I do not want to apply for a new one?",
        "expected_answer": (
            "Yes. A lost or stolen passport must be declared even if the person "
            "does not intend to apply for a new passport."
        )
    },

    {
        "id": 7,
        "language": "en",
        "question": "Do I need to appear in person when applying for a new passport?",
        "expected_answer": (
            "Yes. Lebanese citizens requesting a new passport generally have to "
            "appear personally at the competent regional General Security center."
        )
    },

    {
        "id": 8,
        "language": "en",
        "question": "Can a sick person who cannot leave the hospital be exempted from personal attendance?",
        "expected_answer": (
            "Yes. A sick or paralyzed person unable to leave their residence or hospital "
            "may be exempted, subject to a written request and medical report."
        )
    },

    {
        "id": 9,
        "language": "en",
        "question": "What do I need to certify pages of my passport?",
        "expected_answer": (
            "The passport, a photocopy containing the pages to be certified, "
            "and a tax stamp of LBP 1000."
        )
    },

    {
        "id": 10,
        "language": "en",
        "question": "What documents are required to export a Lebanese passport?",
        "expected_answer": (
            "The required documents include the post-company invoice showing the "
            "sender and receiver addresses and a certified copy of the proxy and "
            "the receiver's passport."
        )
    },


    # =========================================================
    # ARABIC - DRIVING LICENCE
    # =========================================================

    {
        "id": 11,
        "language": "ar",
        "question": "هل يحتاج تجديد رخصة السوق المنتهية الصلاحية إلى موعد؟",
        "expected_answer": "لا، لا يتطلب تجديد رخصة السوق موعداً."
    },

    {
        "id": 12,
        "language": "ar",
        "question": "أين يجب أن أجدد رخصة السوق المنتهية الصلاحية؟",
        "expected_answer": (
            "يجب تجديد رخصة السوق في المركز الذي صدرت عنه."
        )
    },

    {
        "id": 13,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة لتجديد رخصة سوق منتهية الصلاحية؟",
        "expected_answer": (
            "رخصة السوق المنتهية الصلاحية، سجل عدلي حديث، بطاقة أو إفادة فئة الدم، "
            "شهادة طبية حديثة، صورة عن الهوية أو إخراج قيد أو جواز السفر، "
            "إقامة صالحة لغير اللبنانيين، وصورتان شمسيتان مصدقتان."
        )
    },

    {
        "id": 14,
        "language": "ar",
        "question": "ماذا أفعل إذا ضاعت مني رخصة السوق؟",
        "expected_answer": (
            "يمكن إصدار رخصة سوق بديلة بعد تنظيم محضر رسمي لدى قوى الأمن الداخلي "
            "يثبت واقعة فقدان الرخصة."
        )
    },

    {
        "id": 15,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة للحصول على بدل عن رخصة سوق ضائعة غير منتهية الصلاحية؟",
        "expected_answer": (
            "صورة طبق الأصل عن محضر قوى الأمن الداخلي، وصورة عن بطاقة الهوية "
            "أو إخراج قيد أو جواز السفر، وإقامة صالحة لغير اللبنانيين، "
            "وصورتان شمسيتان مصدقتان."
        )
    },

    {
        "id": 16,
        "language": "ar",
        "question": "ماذا أفعل إذا وجدت رخصة السوق القديمة بعد إصدار بدل عن ضائع؟",
        "expected_answer": (
            "يجب تسليم الرخصة القديمة إلى دائرة السوق."
        )
    },

    {
        "id": 17,
        "language": "ar",
        "question": "كيف يمكنني استبدال رخصة سوق أجنبية برخصة سوق لبنانية؟",
        "expected_answer": (
            "يمكن استبدال رخصة السوق الأجنبية برخصة سوق لبنانية وفقاً للأصول "
            "القانونية وبعد التأكد من مطابقة الرخصة الأجنبية للشروط المطلوبة."
        )
    },

    {
        "id": 18,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة لاستبدال رخصة سوق أجنبية برخصة لبنانية؟",
        "expected_answer": (
            "تشمل المستندات إفادة من فرع المرور من مصدر الرخصة مصدقة من وزارة "
            "الخارجية للدولة المعنية، شهادة صحية، صورتين شمسيتين مصدقتين، "
            "إثبات السكن أو الهوية أو الإقامة أو جواز السفر، رخصة السوق الأجنبية، "
            "وسجلاً عدلياً."
        )
    },

    {
        "id": 19,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة للحصول على رخصة سوق دولية؟",
        "expected_answer": (
            "صورة عن جواز السفر، صورة عن الهوية، صورة عن رخصة السوق اللبنانية، "
            "وصورتان شمسيتان."
        )
    },

    {
        "id": 20,
        "language": "ar",
        "question": "ما قيمة الطوابع المطلوبة لاستبدال رخصة سوق لبنانية برخصة سوق أجنبية؟",
        "expected_answer": "50000 ل.ل."
    }
]

In [22]:
def ask(question, k=3):

    # 1. Retrieve relevant chunks
    results = retrieve(question, k=k)

    # 2. Build context for the LLM
    context = "\n\n".join(
    f"""
Service: {result.get('service', '')}
Section: {result['section']}
Language: {result.get('language', '')}
Content:
{result['text']}
"""
    for result in results
)

    # 3. Create the prompt
    messages = [
       {
    "role": "system",
    "content": """
You are an assistant for Lebanese government procedures.

Answer the user's question using ONLY the information contained in the
provided official government context.

IMPORTANT:
- The context may be in Arabic, English, or French.
- The user's question may be in Arabic, English, or French.
- Answer in the same language as the user's question.
- Carefully read information written in Arabic.
- Use relevant information even if the wording of the question is different
  from the wording in the context.
- Do not require an exact phrase match between the question and the context.
- The retrieved context may contain irrelevant passages. Ignore those passages.
- If relevant information is present in any retrieved passage, use it to answer.
- Include all relevant documents, requirements, fees, and conditions found
  in the context.
- Do not invent information that is not present in the context.
- Only say that the information could not be found if NONE of the retrieved
  context contains information that answers the question.

Answer clearly and concisely.
"""
        },
        {
            "role": "user",
            "content": f"""
Official context:

{context}

Question:
{question}
"""
        }
    ]

    # 4. Convert messages into Qwen's chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    # 5. Generate answer
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # 6. Collect unique sources
    sources = []

    for result in results:
        url = result["url"]

        if url and url not in sources:
            sources.append(url)

    return {
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": results
    }

In [23]:
correctness_outputs = []

for test in answerable_questions:

    response = ask(test["question"])

    print("=" * 70)
    print("ID:", test["id"])
    print("Question:", test["question"])
    print("\nExpected:")
    print(test["expected_answer"])
    print("\nGenerated:")
    print(response["answer"])

    correctness_outputs.append({
        "id": test["id"],
        "question": test["question"],
        "expected_answer": test["expected_answer"],
        "generated_answer": response["answer"]
    })

ID: 1
Question: What documents do I need to apply for a biometric passport?

Expected:
The applicant needs the passport application, a Lebanese ID card or civil status extract, a new certified color photo, the old passport if available with copies of its non-empty pages, and the required fees.

Generated:
To apply for a biometric passport, you need:

1. Lebanese ID card OR an extract of civil status (whether this is your first time applying for a biometric passport or not).
2. A new colored photo ID photo on a white background, 4.5 x 3.5, with the name of the individual appearing, as well as the number and place of registered residence, signed and certified by the mayor.
3. Your old passport, if available, along with a copy of the pages that are not empty.
4. Fees related to this application.

Additionally, for minors aged 3 years and under, you need:
- An extract of civil status issued within the last three months.
- A copy of the extract of civil status.
- The above-mentioned documen

In [24]:
correctness_results = [
    True,
    True,
    True,
    True,
    True,
    False,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    False,
    True,
    True,
    True]

In [25]:
correct = sum(correctness_results)
total = len(correctness_results)

print(f"Correctness: {correct}/{total}")
print(f"Correctness percentage: {correct/total:.2%}")

Correctness: 24/26
Correctness percentage: 92.31%


In [26]:
def is_abstention(answer):
    answer = answer.lower()

    honesty_phrases = [
        "could not find",
        "not available in the official sources",
        "لم أجد",
        "غير موجودة في المصادر الرسمية",
        "غير موجود في المصادر الرسمية"
    ]

    return any(phrase in answer for phrase in honesty_phrases)

In [28]:
unanswerable_questions = [

    {
        "id": 1,
        "language": "en",
        "question": "Can I pay my passport fees using Bitcoin?"
    },

    {
        "id": 2,
        "language": "en",
        "question": "Can I pay the passport fee using a credit card?"
    },

    {
        "id": 3,
        "language": "en",
        "question": "How many working days does it take to receive a normal passport?"
    },

    {
        "id": 4,
        "language": "en",
        "question": "Can my new passport be delivered directly to my home?"
    },

    {
        "id": 5,
        "language": "en",
        "question": "What are the opening hours of General Security on Saturday?"
    },

    {
        "id": 6,
        "language": "en",
        "question": "Can I renew my Lebanese passport completely online?"
    },

    {
        "id": 7,
        "language": "ar",
        "question": "هل يمكنني دفع رسوم تجديد رخصة السوق بواسطة بطاقة ائتمان؟"
    },

    {
        "id": 8,
        "language": "ar",
        "question": "كم تستغرق معاملة تجديد رخصة السوق بالأيام؟"
    },

    {
        "id": 9,
        "language": "ar",
        "question": "هل يمكن إرسال رخصة السوق الجديدة إلى منزلي؟"
    },

    {
        "id": 10,
        "language": "ar",
        "question": "ما هي رسوم الحصول على رخصة سوق دولية؟"
    },

    {
        "id": 11,
        "language": "ar",
        "question": "هل يمكن تجديد رخصة السوق بالكامل عبر الإنترنت؟"
    },

    {
        "id": 12,
        "language": "ar",
        "question": "ما هي ساعات دوام مركز تجديد رخص السوق يوم السبت؟"
    }
]

In [29]:
honesty_outputs = []

for test in unanswerable_questions:

    response = ask(test["question"])

    print("=" * 70)
    print("ID:", test["id"])
    print("Question:", test["question"])
    print("Answer:", response["answer"])

    honesty_outputs.append({
        "id": test["id"],
        "question": test["question"],
        "answer": response["answer"]
    })

ID: 1
Question: Can I pay my passport fees using Bitcoin?
Answer: The information provided does not mention any option to pay fees using Bitcoin.
ID: 2
Question: Can I pay the passport fee using a credit card?
Answer: Based on the information provided in the official context, there is no mention of accepting payment via credit card for passport fees. The fees are mentioned to be paid in Lebanese Liras (LBP), but the method of payment is not specified.
ID: 3
Question: How many working days does it take to receive a normal passport?
Answer: The provided context does not specify the number of working days it takes to receive a normal passport. Therefore, the information could not be found.
ID: 4
Question: Can my new passport be delivered directly to my home?
Answer: Based on the information provided in the official context, there is no mention of direct delivery of a new passport to a home address. The context specifies that Lebanese citizens requesting a new passport must personally atte

In [31]:
honesty_results = [
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    False,
    True,
    True,
    True,
    True]

In [32]:
honest = sum(honesty_results)
total = len(honesty_results)

print(f"Correct abstentions: {honest}/{total}")
print(f"Honesty: {honest / total:.2%}")

Correct abstentions: 11/12
Honesty: 91.67%
